In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
import plotly.express as px
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import your CRUD Python module
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Read all records
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectID field if present
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Add Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'

# Safely load image
if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
    logo_component = html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '100px'}
        )
    )
else:
    logo_component = html.Center(html.H5("Logo image not found"))

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H3('Grazioso Salvare Animal Rescue Dashboard')),
    logo_component,
    html.Center(html.H4('Khem Khatiwada')),
    html.Hr(),

    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'RESET'},
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'}
            ],
            value='RESET',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        selected_columns=[],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        dff = pd.DataFrame.from_records(db.read({}))
        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return [
            dcc.Graph(
                figure=px.bar(title='No Data Available')
            )
        ]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.bar(
                breed_counts.head(10),
                x='breed',
                y='count',
                title='Top 10 Breeds in Current Filter'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [
        {
            'if': {'column_id': col},
            'background_color': '#D2F3FF'
        }
        for col in selected_columns
    ]


@app.callback(
    Output('map-id', 'children'),
    Input('datatable-id', 'derived_virtual_data'),
    Input('datatable-id', 'derived_virtual_selected_rows')
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        dff = pd.DataFrame.from_records(db.read({}))
        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return [html.Div("No data available for map")]

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    # Check required columns
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return [html.Div("Location data not available")]

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']

    # Handle missing coordinates
    if pd.isna(lat) or pd.isna(lon):
        return [html.Div("Selected record has no valid map coordinates")]

    breed = dff.iloc[row]['breed'] if 'breed' in dff.columns else 'Animal'
    name = dff.iloc[row]['name'] if 'name' in dff.columns else 'No Name'

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(name))
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server(mode='inline')